# Week 10 Lab — Practical Deep Learning: Augmentation, Dropout & Batch Normalization (Solutions)

---

**What you will build:**
- A cats-vs-dogs binary classifier using CIFAR-10
- Observe overfitting on a baseline model
- Fix it with data augmentation, Dropout, and Batch Normalization
- Compare training curves across all four models

---
# Part 1 — Setup and Data Loading

## Step 1 — Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, Activation
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

## Step 2 — Load CIFAR-10 and Extract Cats & Dogs

In [ ]:
# Load CIFAR-10
(x_train_all, y_train_all), (x_test_all, y_test_all) = cifar10.load_data()

CAT_LABEL = 3
DOG_LABEL = 5

# Extract cats and dogs from training set
train_mask = np.isin(y_train_all.flatten(), [CAT_LABEL, DOG_LABEL])
x_train = x_train_all[train_mask]
y_train_raw = y_train_all[train_mask].flatten()

# Extract cats and dogs from test set
test_mask = np.isin(y_test_all.flatten(), [CAT_LABEL, DOG_LABEL])
x_test = x_test_all[test_mask]
y_test_raw = y_test_all[test_mask].flatten()

# Convert labels to binary: cat=0, dog=1
y_train = (y_train_raw == DOG_LABEL).astype('float32')
y_test  = (y_test_raw  == DOG_LABEL).astype('float32')

print("Training set:", x_train.shape, "| Labels:", y_train.shape)
print("Test set:    ", x_test.shape,  "| Labels:", y_test.shape)
print(f"Cats in train: {np.sum(y_train==0)}, Dogs in train: {np.sum(y_train==1)}")

## Step 3 — Normalize Pixel Values and Visualize

In [ ]:
# Normalize to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

print("Pixel range — min:", x_train.min(), "max:", x_train.max())

In [ ]:
# Visualize some training images
class_names = ['Cat', 'Dog']
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for row, label in enumerate([0, 1]):
    imgs = x_train[y_train == label][:8]
    for col, img in enumerate(imgs):
        axes[row, col].imshow(img)
        axes[row, col].set_title(class_names[label], fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('CIFAR-10 Cats (top) and Dogs (bottom)', y=1.01)
plt.tight_layout()
plt.show()

---
# Part 2 — Baseline CNN (No Augmentation, No Regularization)

## Step 4 — Build the Baseline CNN

In [ ]:
def build_baseline_model():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    return model

baseline_model = build_baseline_model()
baseline_model.summary()

## Step 5 — Compile and Train the Baseline Model

In [ ]:
baseline_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_baseline = baseline_model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=20,
    validation_data=(x_test, y_test),
    verbose=1
)

## Step 6 — Plot Training Curves and Observe Overfitting

In [ ]:
def plot_history(history, title='Training Curves'):
    """Plot training and validation accuracy and loss."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].plot(history.history['accuracy'],     label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title(title + ' — Accuracy')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)
    
    axes[1].plot(history.history['loss'],     label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title(title + ' — Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.show()

plot_history(history_baseline, 'Baseline CNN (no regularization)')

**Answer**: Yes, overfitting is clearly visible. Training accuracy climbs toward ~80–85% while validation accuracy plateaus or drops around epoch 10–15. The gap between them is the signature of overfitting — the model is memorizing training data rather than learning generalizable features.

---
# Part 3 — Data Augmentation

## Step 7 — Visualize Augmented Cat Images

In [ ]:
aug_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2,
    fill_mode='nearest'
)

cat_images = x_train[y_train == 0][:3]

fig, axes = plt.subplots(3, 6, figsize=(13, 7))
for row, cat in enumerate(cat_images):
    axes[row, 0].imshow(cat)
    axes[row, 0].set_title('Original', fontsize=8)
    axes[row, 0].axis('off')
    img_batch = cat.reshape((1,) + cat.shape)
    for col, batch in zip(range(1, 6), aug_datagen.flow(img_batch, batch_size=1)):
        axes[row, col].imshow(np.clip(batch[0], 0, 1))
        axes[row, col].set_title(f'Aug {col}', fontsize=8)
        axes[row, col].axis('off')
plt.suptitle('Cat Images: Original and Augmented Versions', y=1.01)
plt.tight_layout()
plt.show()

## Step 8 — Train the Same Model with Augmentation

In [ ]:
aug_model = build_baseline_model()
aug_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

train_gen = aug_datagen.flow(x_train, y_train, batch_size=64)

history_aug = aug_model.fit(
    train_gen,
    steps_per_epoch=len(x_train) // 64,
    epochs=20,
    validation_data=(x_test, y_test),
    verbose=1
)

plot_history(history_aug, 'CNN with Data Augmentation')

**Answer**: Augmentation reduces overfitting by making the model see more varied versions of each image. Training accuracy grows more slowly (the problem is harder) while validation accuracy is more stable. The train/val gap is smaller than the baseline.

---
# Part 4 — Adding Dropout

## Step 9 — Build a Model with Dropout

In [ ]:
def build_dropout_model():
    model = Sequential([
        Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
        MaxPooling2D(2,2),
        Conv2D(64, (3,3), activation='relu'),
        MaxPooling2D(2,2),
        Dropout(0.25),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model

dropout_model = build_dropout_model()
dropout_model.summary()

## Step 10 — Train with Augmentation + Dropout

In [ ]:
dropout_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

train_gen2 = aug_datagen.flow(x_train, y_train, batch_size=64)
history_dropout = dropout_model.fit(
    train_gen2,
    steps_per_epoch=len(x_train) // 64,
    epochs=20,
    validation_data=(x_test, y_test),
    verbose=1
)

plot_history(history_dropout, 'CNN with Augmentation + Dropout')

---
# Part 5 — Adding Batch Normalization

## Step 11 — Build a Model with Batch Normalization + Dropout

In [ ]:
def build_bn_model():
    model = Sequential([
        # Block 1
        Conv2D(32, (3,3), input_shape=(32,32,3)),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(2,2),

        # Block 2
        Conv2D(64, (3,3)),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(2,2),
        Dropout(0.25),

        # Dense head
        Flatten(),
        Dense(128),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model

bn_model = build_bn_model()
bn_model.summary()

## Step 12 — Train with Augmentation + BatchNorm + Dropout

In [ ]:
bn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

train_gen3 = aug_datagen.flow(x_train, y_train, batch_size=64)
history_bn = bn_model.fit(
    train_gen3,
    steps_per_epoch=len(x_train) // 64,
    epochs=20,
    validation_data=(x_test, y_test),
    verbose=1
)

plot_history(history_bn, 'CNN with Augmentation + BatchNorm + Dropout')

---
# Part 6 — Compare All Models

## Step 13 — Side-by-Side Validation Accuracy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history_baseline.history['val_accuracy'], label='Baseline',               linestyle='--')
ax.plot(history_aug.history['val_accuracy'],      label='+ Augmentation',         linestyle='-')
ax.plot(history_dropout.history['val_accuracy'],  label='+ Augmentation+Dropout', linestyle='-.')
ax.plot(history_bn.history['val_accuracy'],       label='+ Aug+BN+Dropout',       linestyle=':')

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('Validation Accuracy: All Four Models')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("Final validation accuracies (last epoch):")
print(f"  Baseline:               {history_baseline.history['val_accuracy'][-1]:.3f}")
print(f"  + Augmentation:         {history_aug.history['val_accuracy'][-1]:.3f}")
print(f"  + Aug + Dropout:        {history_dropout.history['val_accuracy'][-1]:.3f}")
print(f"  + Aug + BN + Dropout:   {history_bn.history['val_accuracy'][-1]:.3f}")

---
# Part 7 — Reflection (Model Answers)

**1. Which model shows the most overfitting?**  
The **Baseline** model overfits the most. The training accuracy climbs steeply while the validation accuracy plateaus and the loss diverges — a classic overfitting signature.

**2. Did augmentation alone help?**  
Yes. By exposing the model to rotated, shifted, and flipped versions of the training images, augmentation makes it harder to memorize exact pixel patterns. The train/val accuracy gap is visibly reduced.

**3. What effect did Dropout have?**  
Dropout lowers training accuracy (the network is harder to train) but improves or stabilizes validation accuracy. The training curve becomes noisier but the val curve is smoother and higher than the baseline.

**4. Does BatchNorm speed up training?**  
Yes — the model with BatchNorm typically reaches a comparable or better validation accuracy in fewer epochs because the normalized inputs allow the optimizer to find good weights faster.

**5. Which model would you deploy?**  
The **Aug + BN + Dropout** model: it has the best generalization, most stable training, and all techniques are complementary with no significant cost at inference time.

**6. Bonus — 10-class CIFAR-10**  
Changes needed:
- Output layer: `Dense(10, activation='softmax')` instead of `Dense(1, activation='sigmoid')`
- Loss: `categorical_crossentropy` instead of `binary_crossentropy`
- Labels: one-hot encode with `to_categorical(y, 10)` instead of binary 0/1

```python
# Output layer
Dense(10, activation='softmax')

# Compile
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Labels
y_train_cat = to_categorical(y_train_all, 10)
y_test_cat  = to_categorical(y_test_all,  10)
```